In [ ]:
# -*- coding: utf-8 -*-
"""
@author: Hiromasa Kaneko　https://github.com/hkaneko1985/python_doe_kspub.
Modified by Masanori Kodera　https://github.com/masanorikodera
"""

import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
number_of_test_samples = 10 
number_of_trees = 300 

#dataset = pd.read_csv('dataset_region1.csv', index_col=0, header=0)
dataset = pd.read_csv('dataset_region1.csv', index_col=0, header=0)
dataset.head()

In [ ]:
dataset=dataset.dropna()
dataset.shape

In [ ]:
y = dataset.iloc[:, -1]  
x = dataset.iloc[:, :-1] 


if number_of_test_samples == 0:
    x_train = x.copy()
    x_test = x.copy()
    y_train = y.copy()
    y_test = y.copy()
else:
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=number_of_test_samples, shuffle=True,
                                                        random_state=12345
                                                       )

deleting_variables = x_train.columns[x_train.std() == 0]
x_train = x_train.drop(deleting_variables, axis=1)
x_test = x_test.drop(deleting_variables, axis=1)

In [ ]:
model = RandomForestRegressor(n_estimators=number_of_trees,
                              max_features=4,
                              oob_score=True) 
model.fit(x_train, y_train)  

In [ ]:
variable_importances = pd.DataFrame(model.feature_importances_, index=x_train.columns, columns=['importances']) 
variable_importances.to_csv(
    'variable_importances_rf.csv') 

estimated_y_train = model.predict(x_train)  
estimated_y_train = pd.DataFrame(estimated_y_train, index=x_train.index, columns=['estimated_y'])


plt.rcParams['font.size'] = 18
plt.scatter(y_train, estimated_y_train.iloc[:, 0], c='blue')  
y_max = max(y_train.max(), estimated_y_train.iloc[:, 0].max()) 
y_min = min(y_train.min(), estimated_y_train.iloc[:, 0].min())  
plt.plot([y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)],
         [y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)], 'k-')  
plt.ylim(y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min))  
plt.xlim(y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)) 
plt.xlabel('actual y') 
plt.ylabel('estimated y') 
plt.gca().set_aspect('equal', adjustable='box')  
plt.show() 

print('r^2 for training data :', r2_score(y_train, estimated_y_train))
print('RMSE for training data :', mean_squared_error(y_train, estimated_y_train, squared=False))
print('MAE for training data :', mean_absolute_error(y_train, estimated_y_train))

y_train_for_save = pd.DataFrame(y_train)
y_train_for_save.columns = ['actual_y']
y_error_train = y_train_for_save.iloc[:, 0] - estimated_y_train.iloc[:, 0]
y_error_train = pd.DataFrame(y_error_train)
y_error_train.columns = ['error_of_y(actual_y-estimated_y)']
results_train = pd.concat([y_train_for_save, estimated_y_train, y_error_train], axis=1) 
results_train.to_csv('estimated_y_train_in_detail_rf.csv') 

In [ ]:
estimated_y_test = model.predict(x_test)  
estimated_y_test = pd.DataFrame(estimated_y_test, index=x_test.index, columns=['estimated_y'])

plt.rcParams['font.size'] = 18
plt.scatter(y_test, estimated_y_test.iloc[:, 0], c='blue')  
y_max = max(y_test.max(), estimated_y_test.iloc[:, 0].max())  
y_min = min(y_test.min(), estimated_y_test.iloc[:, 0].min()) 
plt.plot([y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)],
         [y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)], 'k-')  
plt.ylim(y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min)) 
plt.xlim(y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min))  
plt.xlabel('actual y') 
plt.ylabel('estimated y') 
plt.gca().set_aspect('equal', adjustable='box')  
plt.show()  

print('r^2 for test data :', r2_score(y_test, estimated_y_test))
print('RMSE for test data :', mean_squared_error(y_test, estimated_y_test, squared=False))
print('MAE for test data :', mean_absolute_error(y_test, estimated_y_test))


y_test_for_save = pd.DataFrame(y_test)
y_test_for_save.columns = ['actual_y']
y_error_test = y_test_for_save.iloc[:, 0] - estimated_y_test.iloc[:, 0]
y_error_test = pd.DataFrame(y_error_test)
y_error_test.columns = ['error_of_y(actual_y-estimated_y)']
results_test = pd.concat([y_test_for_save, estimated_y_test, y_error_test], axis=1)
results_test.to_csv('estimated_y_test_in_detail_rf.csv') 